In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pprint
import joblib
from skimage.feature import hog
from skimage.feature import local_binary_pattern
import cv2
import os

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
IMG_SIZE = (128,128)
TEST_SIZE = 0.2

In [1]:
# =========================================================
# CELL : EXTRACT ZIP DATASET
# =========================================================

import os
import zipfile

# ZIP file path
ZIP_FILE = "archive.zip"

# Folder where dataset will be extracted
DATASET_FOLDER = "dataset"

# Create folder
os.makedirs(DATASET_FOLDER, exist_ok=True)

# Extract ZIP
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(DATASET_FOLDER)

print("ZIP Extracted Successfully")



ZIP Extracted Successfully


In [4]:
# Dataset path for further use
dataset_path = "dataset/data"

print("Dataset Path :", dataset_path)

# =========================================================
# PRINT CLASSES NAME
# =========================================================

classes = os.listdir(dataset_path)

print("\nClasses Found :")

for class_name in classes:
    print(class_name)

Dataset Path : dataset/data

Classes Found :
with_mask
without_mask


In [5]:
def extract_features(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.resize(img,IMG_SIZE)
    gray = cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)
    feats = []


    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    for ch in cv2.split(hsv):
        feats += [ch.mean(), ch.std()]

    #feats.append(cv2.Laplacian(gray, cv2.CV_64F).var())

    lbp = local_binary_pattern(gray,P=8, R=1, method = "uniform")
    hist, _ = np.histogram(lbp, bins = 10, range=(0,10), density = True)
    feats += list(hist)

    edges = cv2.Canny(gray,100,200)
    feats.append(edges.mean())

    moments = cv2.moments(gray)
    hu = cv2.HuMoments(moments).flatten()
    feats+= list(-np.sign(hu)*np.log10(np.abs(hu) + 1e-10))

    cnts, _ = cv2.findContours(edges,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        c = max(cnts,key=cv2.contourArea)
        feats += [cv2.contourArea(c), cv2.arcLength(c,True)]
    else:
        feats += [0,0]

    hog_feats = hog(gray, orientations = 9, pixels_per_cell = (8,8), cells_per_block = (2,2), feature_vector=True)
    feats += list(hog_feats)

    return np.array(feats, dtype=np.float32)
    

In [6]:
X,y = [],[]
with_mask = "dataset/data/with_mask"
for file in os.listdir(with_mask):
    path = os.path.join(with_mask,file)
    f = extract_features(path)
    if f is not None:
        X.append(f)
        y.append("withmask")
    else:
        print(f"skip corrupted{file}")
without_mask= "dataset/data/without_mask"
for file in os.listdir(without_mask):
    path = os.path.join(without_mask,file)
    f = extract_features(path)
    if f is not None:
        X.append(f)
        y.append("withoutmask")
    else:
        print(f"skip corrupted{file}")

In [7]:
X = np.array(X)
y = np.array(y)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = TEST_SIZE, random_state=42, stratify=y)

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
clf = SVC(kernel='linear')
clf.fit(X_train, y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [8]:
clf = RandomForestClassifier(n_estimators = 100, random_state=42)
clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [11]:
y_pred = clf.predict(X_test)

classes = ["withmask", "withoutmask"]

# Classification Report
print(classification_report(y_test, y_pred, target_names=classes))

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

              precision    recall  f1-score   support

    withmask       0.83      0.83      0.83       745
 withoutmask       0.84      0.84      0.84       766

    accuracy                           0.83      1511
   macro avg       0.83      0.83      0.83      1511
weighted avg       0.83      0.83      0.83      1511

Accuracy: 83.45%


In [12]:
from sklearn.svm import LinearSVC
clf = LinearSVC(C=1.0, max_iter=2000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["withmask","withoutmask"]))
# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


              precision    recall  f1-score   support

    withmask       0.83      0.83      0.83       745
 withoutmask       0.84      0.84      0.84       766

    accuracy                           0.84      1511
   macro avg       0.84      0.84      0.84      1511
weighted avg       0.84      0.84      0.84      1511

Accuracy: 83.59%
